# Sticky Regime Decomposition
## Identifier quel régime de stickiness explique le mouvement du smile — Actions US

---

## Problème des traders

Entre hier ($t_0$) et aujourd'hui ($t_1$), la surface de volatilité implicite s'est déformée.  
Les traders veulent savoir : **quel mélange de règles sticky** explique cette déformation ?

Les trois régimes purs :

| Régime | Définition | SSR de Bergomi |
|---|---|---|
| **Sticky Delta** | Le smile en moneyness $(m=K/S, \tau)$ reste fixe | $R = 0$ |
| **Sticky Strike** | La vol pour un strike absolu $(K,T)$ reste fixe | $R = 1$ |
| **Sticky Skew** | La vol ATM se déplace de $2 \times$ skew $\times \Delta\ln S$ | $R = 2$ |

## Pourquoi l'OLS jour-à-jour ne fonctionne pas

Quand $\Delta S \approx 0$, les régresseurs s'effondrent → R² catastrophique.  
**Solution :** régression sur une fenêtre glissante de $N$ jours — on estime $\beta(m)$ = réponse historique de la vol au spot.

## Plan du notebook

| Section | Contenu |
|---|---|
| **0** | Setup & connexion MDX |
| **1** | Téléchargement & construction des surfaces US |
| **2** | Analyse descriptive : surface moyenne, std, skew |
| **3** | Régression glissante $\Delta\hat{\sigma} = \alpha + \beta \cdot \Delta\ln S + \varepsilon$ |
| **4** | Courbe $\beta(m)$ — signature du régime |
| **5** | SSR normalisé $\tilde{\beta}(m)$ — comparaison aux régimes purs |
| **6** | Décomposition polynomiale $(w_0, w_1, w_2)$ — niveau, asymétrie, courbure |
| **7** | Décomposition quotidienne : part spot vs part autonome |
| **8** | Analyse temporelle : évolution du régime dans le temps |
| **9** | Comparaison multi-ticker & multi-maturité |
| **10** | Interface trader : vue synthétique |
| **11** | Validation croisée & robustesse |


---
## Section 0 — Setup & connexion MDX

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import warnings, pickle
from pathlib import Path
from itertools import product as iproduct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from scipy.stats import norm, linregress
from scipy.linalg import lstsq
from scipy.optimize import brentq
from statsmodels.tsa.stattools import acf
from pandas.tseries.offsets import BDay

warnings.simplefilter('ignore')

plt.rcParams.update({
    'figure.figsize': (13, 5),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

cache_dir = Path('./cache_sticky')
cache_dir.mkdir(exist_ok=True)

print('Imports OK')

In [ ]:
# ============================================================
#  CONNEXION MDX
# ============================================================
import ezmdx
from maxxpy.apis.mdx.api import MdxClient

LOGIN_MDX    = ""    # <-- TON LOGIN
PASSWORD_MDX = ""    # <-- TON MOT DE PASSE

MDX_TYPES = {
    'volatility': 'EQUITY_VOLATILITY',
    'spot':       ['STOCK_QUOTE', 'INDEX_QUOTE', 'FUND_QUOTE'],
}

ezmdx.set_app(app_name='VEGA5')
ezmdx.prod.satis_login()

mtx_client = MdxClient('MSD', LOGIN_MDX, PASSWORD_MDX, use_prod_only=True)
print('Connexion MDX OK')

In [ ]:
# ============================================================
#  PARAMÈTRES GLOBAUX — MODIFIER ICI
# ============================================================

# ─── Univers d'actions US ──────────────────────────────────
TICKERS = [
    'AAPL',   # Apple
    'MSFT',   # Microsoft
    'NVDA',   # Nvidia
    'SPY',    # S&P 500 ETF (référence)
    'TSLA',   # Tesla (vol élevée)
    'AMZN',   # Amazon
]

# ─── Horizons d'analyse ────────────────────────────────────
TODAY        = pd.Timestamp.today().normalize()
DATE_END     = TODAY - BDay(1)           # hier
DATE_START   = DATE_END - pd.DateOffset(years=2)  # 2 ans d'historique

# ─── Grille cible (m, τ) ───────────────────────────────────
# Moneyness : de 80% à 120% par pas de 5%
M_GRID   = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])
# Maturités cibles en années
TAU_GRID = np.array([1/12., 2/12., 3/12., 6/12., 9/12., 12/12.])

# ─── Fenêtres de régression ────────────────────────────────
ROLL_WINDOWS = {
    '1M':  21,
    '3M':  63,
    '6M': 126,
}
ROLL_DEFAULT = '3M'  # fenêtre utilisée par défaut

# ─── Bandes passantes Nadaraya-Watson ──────────────────────
H1 = 0.08   # moneyness
H2 = 0.12   # maturité

print(f'Période : {DATE_START.date()} → {DATE_END.date()}')
print(f'Tickers : {TICKERS}')
print(f'Grille  : {len(M_GRID)} moneyness × {len(TAU_GRID)} maturités')

---
## Section 1 — Téléchargement & construction des surfaces

In [ ]:
# ============================================================
#  FONCTIONS DE FETCH MDX
# ============================================================
def fetch_vols(ticker, date_start, date_end, mtx_client, asset_type='S'):
    """Récupère les vols implicites pour un ticker US."""
    all_bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    code = f'{asset_type}_{ticker}'
    try:
        df = mtx_client.get_market_data(
            mdx_type=MDX_TYPES['volatility'], code=code, date=all_bdays
        )
        return df[['STRIKE', 'MATURITY', 'VOLATILITY', 'DATE']].copy()
    except Exception as e:
        print(f'  Erreur vols {ticker} : {e}')
        return None


def fetch_spot(ticker, date_start, date_end, mtx_client):
    """Récupère le spot pour un ticker US."""
    all_bdays = pd.bdate_range(date_start - BDay(3), date_end + BDay(3)).strftime('%Y-%m-%d').tolist()
    for mdx_type in MDX_TYPES['spot']:
        try:
            df = mtx_client.get_market_data(
                mdx_type=mdx_type, code=ticker, date=all_bdays
            )
            df['date'] = pd.to_datetime(df['DATE'])
            price_col = [c for c in df.columns if c != 'DATE'][0]
            df['spot'] = pd.to_numeric(df[price_col], errors='coerce')
            return df[['date', 'spot']].dropna().set_index('date')['spot']
        except Exception:
            continue
    return None


# ============================================================
#  TÉLÉCHARGEMENT AVEC CACHE
# ============================================================
def load_or_fetch(ticker, date_start, date_end, mtx_client, force=False):
    cache_path = cache_dir / f'{ticker}_raw.pkl'
    if cache_path.exists() and not force:
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f'Fetching {ticker}...')
    vols = fetch_vols(ticker, date_start, date_end, mtx_client)
    spot = fetch_spot(ticker, date_start, date_end, mtx_client)

    if vols is not None and spot is not None:
        data = {'vols': vols, 'spot': spot}
        with open(cache_path, 'wb') as f:
            pickle.dump(data, f)
        print(f'  → {len(vols):,} quotes, {spot.dropna().__len__()} jours spot')
        return data
    return None


# Téléchargement de tous les tickers
RAW_DATA = {}
for ticker in TICKERS:
    data = load_or_fetch(ticker, DATE_START, DATE_END, mtx_client)
    if data is not None:
        RAW_DATA[ticker] = data

print(f'\nDonnées disponibles : {list(RAW_DATA.keys())}')

In [ ]:
# ============================================================
#  CONSTRUCTION DES SURFACES LISSÉES (Nadaraya-Watson)
# ============================================================
def bs_price(S, K, tau, sigma, r=0., q=0., option='call'):
    if tau <= 1e-6 or sigma <= 1e-6: return max(S-K,0.) if option=='call' else max(K-S,0.)
    d1 = (np.log(S/K) + tau*(r-q+0.5*sigma**2)) / (sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    disc = np.exp(-r*tau)
    if option=='call': return S*np.exp(-q*tau)*norm.cdf(d1) - K*disc*norm.cdf(d2)
    return K*disc*norm.cdf(-d2) - S*np.exp(-q*tau)*norm.cdf(-d1)

def implied_vol(S, K, tau, price, r=0., q=0., option='call'):
    try:
        return brentq(lambda v: bs_price(S,K,tau,v,r,q,option)-price, 1e-4, 5., xtol=1e-8)
    except: return np.nan


def prepare_daily(vols_raw, spot_series, r=0., q=0.):
    """Prépare les données journalières en moneyness/maturité."""
    df = vols_raw.rename(columns={
        'STRIKE':'K','MATURITY':'mat','VOLATILITY':'iv','DATE':'date'})
    df['date'] = pd.to_datetime(df['date'])
    df['mat']  = pd.to_datetime(df['mat'])
    df['K']    = pd.to_numeric(df['K'],  errors='coerce')
    df['iv']   = pd.to_numeric(df['iv'], errors='coerce')
    if df['iv'].median() > 2: df['iv'] /= 100.

    spot_df = spot_series.reset_index()
    spot_df.columns = ['date', 'S']
    spot_df['date'] = pd.to_datetime(spot_df['date'])

    df = df.merge(spot_df, on='date', how='left').dropna()
    df['bdays'] = [len(pd.bdate_range(d, m))-1 for d,m in zip(df['date'], df['mat'])]
    df = df[(df['bdays'] > 0) & (df['bdays'] < 400)]
    df['tau'] = df['bdays'] / 252.
    df['m']   = df['K'] / df['S']

    # Filtres qualité
    df = df[(df['m'] >= 0.70) & (df['m'] <= 1.30)]
    df = df[(df['tau'] >= 0.05) & (df['tau'] <= 1.5)]
    df = df[(df['iv'] > 0.01) & (df['iv'] < 3.0)]
    return df.sort_values(['date','tau','m']).reset_index(drop=True)


def nadaraya_watson_surface(day_df, m_grid, tau_grid, h1, h2):
    """Lisse la surface d'un jour sur la grille fixe (m_grid × tau_grid)."""
    surf = np.full((len(m_grid), len(tau_grid)), np.nan)
    m_obs   = day_df['m'].values
    tau_obs = day_df['tau'].values
    iv_obs  = day_df['iv'].values

    for i, m in enumerate(m_grid):
        for j, tau in enumerate(tau_grid):
            w = np.exp(-((m_obs-m)**2)/(2*h1**2) - ((tau_obs-tau)**2)/(2*h2**2))
            denom = w.sum()
            if denom > 1e-12:
                surf[i, j] = (w * iv_obs).sum() / denom
    return surf


def build_surface_ts(ticker_data, m_grid, tau_grid, h1, h2, min_obs=8):
    """
    Construit la série temporelle de surfaces lissées pour un ticker.
    Retourne un DataFrame MultiIndex (date, m, tau) → iv
    """
    df = prepare_daily(ticker_data['vols'], ticker_data['spot'])
    dates = sorted(df['date'].unique())

    records = []
    for date in dates:
        day = df[df['date'] == date]
        if len(day) < min_obs:
            continue
        S = day['S'].iloc[0]
        surf = nadaraya_watson_surface(day, m_grid, tau_grid, h1, h2)

        # Remplissage des NaN
        surf = pd.DataFrame(surf).interpolate(axis=0).interpolate(axis=1).values
        if np.isnan(surf).sum() > surf.size * 0.4:
            continue

        for i, m in enumerate(m_grid):
            for j, tau in enumerate(tau_grid):
                iv = surf[i, j]
                if np.isfinite(iv) and iv > 0.01:
                    records.append({'date': date, 'm': m, 'tau': tau,
                                    'iv': iv, 'S': S})

    result = pd.DataFrame(records)
    result['date'] = pd.to_datetime(result['date'])
    return result.set_index(['date','m','tau']).sort_index()


# Construction avec cache par ticker
SURFACES = {}
for ticker, data in RAW_DATA.items():
    surf_cache = cache_dir / f'{ticker}_surfaces.pkl'
    if surf_cache.exists():
        with open(surf_cache, 'rb') as f:
            SURFACES[ticker] = pickle.load(f)
        print(f'{ticker} : surfaces chargées ({len(SURFACES[ticker])} points)')
    else:
        print(f'Construction surfaces {ticker}...')
        surf = build_surface_ts(data, M_GRID, TAU_GRID, H1, H2)
        SURFACES[ticker] = surf
        with open(surf_cache, 'wb') as f:
            pickle.dump(surf, f)
        print(f'  → {len(surf)} points')

---
## Section 2 — Analyse descriptive : profil moyen, std, skew observé

In [ ]:
# ============================================================
#  STATISTIQUES DESCRIPTIVES PAR TICKER
# ============================================================
def get_surface_matrix(surf_df, tau_target, tau_tol=0.02):
    """
    Extrait la série temporelle de la surface pour une maturité cible.
    Retourne : DataFrame (dates × moneyness) de vols implicites
    """
    avail_tau = surf_df.index.get_level_values('tau').unique()
    closest   = avail_tau[np.argmin(np.abs(avail_tau - tau_target))]
    if abs(closest - tau_target) > tau_tol:
        return None
    sub = surf_df.xs(closest, level='tau')
    pivot = sub['iv'].unstack('m')
    return pivot, closest


def get_spot_series(ticker):
    spot = RAW_DATA[ticker]['spot']
    spot.index = pd.to_datetime(spot.index)
    return spot.sort_index()


# ─── Vue d'ensemble pour un ticker ─────────────────────────
TICKER_FOCUS = 'AAPL'   # ticker pour l'analyse détaillée
TAU_FOCUS    = 3/12.    # maturité cible : 3 mois

surf_focus = SURFACES[TICKER_FOCUS]
mat_ts, tau_actual = get_surface_matrix(surf_focus, TAU_FOCUS)
spot_ts    = get_spot_series(TICKER_FOCUS)

print(f'Analyse détaillée : {TICKER_FOCUS} | τ ≈ {tau_actual*12:.1f}M')
print(f'N jours disponibles : {len(mat_ts)}')
print(f'Plage de dates : {mat_ts.index.min().date()} → {mat_ts.index.max().date()}')

In [ ]:
# ============================================================
#  FIGURE 1 — PROFIL DESCRIPTIF DU SMILE
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1a. Surface moyenne
mean_smile  = mat_ts.mean()
std_smile   = mat_ts.std()

axes[0, 0].fill_between(
    M_GRID,
    (mean_smile - std_smile).values * 100,
    (mean_smile + std_smile).values * 100,
    alpha=0.25, color='steelblue', label='±1 std'
)
axes[0, 0].plot(M_GRID, mean_smile.values * 100, 'steelblue', lw=2.5, label='Moyenne')
axes[0, 0].axvline(1., ls='--', color='gray', lw=1.)
axes[0, 0].set_xlabel('Moneyness m')
axes[0, 0].set_ylabel('Vol implicite (%)')
axes[0, 0].set_title(f'{TICKER_FOCUS} — Smile moyen (τ≈{tau_actual*12:.0f}M)')
axes[0, 0].legend(fontsize=9)

# 1b. Série temporelle ATM
atm_idx  = np.argmin(np.abs(M_GRID - 1.0))
atm_ts   = mat_ts.iloc[:, atm_idx]
axes[0, 1].plot(atm_ts.index, atm_ts.values * 100, 'steelblue', lw=1.5)
axes[0, 1].set_ylabel('Vol ATM (%)')
axes[0, 1].set_title(f'Vol ATM dans le temps (τ≈{tau_actual*12:.0f}M)')
axes[0, 1].xaxis.set_tick_params(rotation=30)

# 1c. Skew 90-110 dans le temps
m90_idx  = np.argmin(np.abs(M_GRID - 0.90))
m110_idx = np.argmin(np.abs(M_GRID - 1.10))
skew_ts  = mat_ts.iloc[:, m90_idx] - mat_ts.iloc[:, m110_idx]
axes[0, 2].plot(skew_ts.index, skew_ts.values * 100, 'firebrick', lw=1.5)
axes[0, 2].axhline(0., color='k', lw=0.7)
axes[0, 2].set_ylabel('Skew 90-110 (vol pts)')
axes[0, 2].set_title('Skew 90%-110% dans le temps')
axes[0, 2].xaxis.set_tick_params(rotation=30)

# 1d. Std des variations log-quotidiennes par moneyness
log_var  = np.log(mat_ts).diff().dropna()
std_lv   = log_var.std()
axes[1, 0].bar(M_GRID, std_lv.values * 100, width=0.04,
               color='steelblue', alpha=0.8, edgecolor='k', lw=0.5)
axes[1, 0].set_xlabel('Moneyness m')
axes[1, 0].set_ylabel('Std log-var quotidien (%)')
axes[1, 0].set_title('Intensité des fluctuations journalières\n(test sticky moneyness)')

# 1e. Distribution des variations log-vol ATM
dlog_atm = log_var.iloc[:, atm_idx].dropna()
axes[1, 1].hist(dlog_atm.values * 100, bins=60,
               color='steelblue', alpha=0.8, density=True, edgecolor='k', lw=0.3)
x_g = np.linspace(dlog_atm.min()*100, dlog_atm.max()*100, 200)
axes[1, 1].plot(x_g, norm.pdf(x_g, dlog_atm.mean()*100, dlog_atm.std()*100),
               'firebrick', lw=2, label='Normale')
axes[1, 1].set_xlabel('Δ log(vol ATM) (%)')
axes[1, 1].set_ylabel('Densité')
axes[1, 1].set_title(f'Distribution Δ log(vol ATM)\n'
                     f'Skew={dlog_atm.skew():.2f}, Kurt={dlog_atm.kurtosis():.2f}')
axes[1, 1].legend(fontsize=9)

# 1f. Corrélation vol ATM / rendement spot
spot_aligned = spot_ts.reindex(atm_ts.index, method='nearest')
dlog_spot    = np.log(spot_aligned).diff().dropna()
dlog_vol     = atm_ts.diff().dropna()
common_idx   = dlog_spot.index.intersection(dlog_vol.index)
axes[1, 2].scatter(
    dlog_spot.loc[common_idx].values * 100,
    dlog_vol.loc[common_idx].values  * 100,
    alpha=0.3, s=8, color='steelblue'
)
slope_atm, intercept_atm, r_val, _, _ = linregress(
    dlog_spot.loc[common_idx].values,
    dlog_vol.loc[common_idx].values
)
x_line = np.linspace(dlog_spot.loc[common_idx].min(), dlog_spot.loc[common_idx].max(), 100)
axes[1, 2].plot(x_line * 100, (slope_atm * x_line + intercept_atm) * 100,
               'firebrick', lw=2.5, label=f'β={slope_atm:.3f}\nR²={r_val**2:.3f}')
axes[1, 2].set_xlabel('Δ ln(S) (%)')
axes[1, 2].set_ylabel('Δ σ_ATM (%)')
axes[1, 2].set_title('Levier : vol ATM vs rendement spot')
axes[1, 2].legend(fontsize=9)

plt.suptitle(f'Section 2 — Analyse descriptive : {TICKER_FOCUS} | τ ≈ {tau_actual*12:.0f}M',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 3 — Régression glissante $\Delta\hat{\sigma} = \alpha + \beta \cdot \Delta\ln S + \varepsilon$

### Pourquoi la fenêtre glissante résout le problème

Au lieu de régresser sur un seul jour (où $\Delta S \approx 0$ est catastrophique), on régresse sur $N$ jours. Les estimateurs $\hat{\beta}(m)$ sont stables car ils agrègent de nombreux jours avec des $\Delta S$ variés.

Pour chaque point de moneyness $m_i$ et une fenêtre $[t-N, t]$ :
$$\Delta\hat{\sigma}_s(m_i) = \alpha_i + \beta_i \cdot \Delta\ln S_s + \varepsilon_{s,i}, \qquad s = t-N, \ldots, t$$

On obtient la **courbe $\beta(m)$** = comment la vol à la moneyness $m$ répond au spot, en moyenne sur la fenêtre.

In [ ]:
# ============================================================
#  RÉGRESSION GLISSANTE POINT PAR POINT EN MONEYNESS
# ============================================================
def compute_rolling_beta(mat_ts, spot_ts, window=63):
    """
    Calcule β(m, t) par régression glissante sur 'window' jours.

    Pour chaque date t et chaque moneyness m_i :
        Δσ_s(m_i) = α_i + β_i * ΔlnS_s + ε    sur s ∈ [t-window, t]

    Retourne
    --------
    beta_df   : DataFrame (dates × moneyness) — β(m, t)
    alpha_df  : DataFrame (dates × moneyness) — α(m, t) (drift autonome)
    r2_df     : DataFrame (dates × moneyness) — R²(m, t)
    se_df     : DataFrame (dates × moneyness) — std erreur de β
    """
    # Aligner spot et surface
    spot_aligned = spot_ts.reindex(mat_ts.index, method='nearest').dropna()
    common_idx   = mat_ts.index.intersection(spot_aligned.index)
    mat_ts_c     = mat_ts.loc[common_idx]
    spot_c       = spot_aligned.loc[common_idx]

    # Variations
    d_vol  = mat_ts_c.diff().dropna()               # Δσ (en points de vol)
    d_spot = np.log(spot_c).diff().dropna()          # Δ ln S
    common2 = d_vol.index.intersection(d_spot.index)
    d_vol   = d_vol.loc[common2]
    d_spot  = d_spot.loc[common2]

    dates     = d_vol.index
    n_dates   = len(dates)
    m_cols    = d_vol.columns

    beta_vals  = np.full((n_dates, len(m_cols)), np.nan)
    alpha_vals = np.full((n_dates, len(m_cols)), np.nan)
    r2_vals    = np.full((n_dates, len(m_cols)), np.nan)
    se_vals    = np.full((n_dates, len(m_cols)), np.nan)

    for t in range(window, n_dates):
        # Fenêtre glissante
        y_mat = d_vol.iloc[t-window:t].values      # (window, n_m)
        x_vec = d_spot.iloc[t-window:t].values     # (window,)
        X     = np.column_stack([np.ones(window), x_vec])  # design matrix

        for j in range(len(m_cols)):
            y = y_mat[:, j]
            mask = np.isfinite(y) & np.isfinite(x_vec)
            if mask.sum() < window // 2:
                continue
            try:
                slope, intercept, r_val, _, se = linregress(x_vec[mask], y[mask])
                beta_vals[t, j]  = slope
                alpha_vals[t, j] = intercept
                r2_vals[t, j]    = r_val**2
                se_vals[t, j]    = se
            except Exception:
                pass

    beta_df  = pd.DataFrame(beta_vals,  index=dates, columns=m_cols)
    alpha_df = pd.DataFrame(alpha_vals, index=dates, columns=m_cols)
    r2_df    = pd.DataFrame(r2_vals,    index=dates, columns=m_cols)
    se_df    = pd.DataFrame(se_vals,    index=dates, columns=m_cols)

    return beta_df, alpha_df, r2_df, se_df


# Calcul pour le ticker focus, toutes les fenêtres
BETA_RESULTS = {}
for win_label, win_size in ROLL_WINDOWS.items():
    print(f'Calcul β(m,t) — fenêtre {win_label} ({win_size}j)...')
    beta_df, alpha_df, r2_df, se_df = compute_rolling_beta(
        mat_ts, spot_ts, window=win_size
    )
    BETA_RESULTS[win_label] = {
        'beta': beta_df, 'alpha': alpha_df,
        'r2':   r2_df,   'se':    se_df
    }
    print(f'  → {beta_df.dropna().shape[0]} dates avec résultats')

# Référence : fenêtre par défaut
beta_df  = BETA_RESULTS[ROLL_DEFAULT]['beta']
alpha_df = BETA_RESULTS[ROLL_DEFAULT]['alpha']
r2_df    = BETA_RESULTS[ROLL_DEFAULT]['r2']
se_df    = BETA_RESULTS[ROLL_DEFAULT]['se']

In [ ]:
# ============================================================
#  DIAGNOSTICS DE LA RÉGRESSION
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R² moyen par moneyness
r2_mean = r2_df.mean()
axes[0].bar(r2_mean.index.astype(float), r2_mean.values,
            width=0.04, color='steelblue', alpha=0.8, edgecolor='k', lw=0.5)
axes[0].axhline(0.5, ls='--', color='firebrick', lw=1.5, label='R²=50%')
axes[0].set_xlabel('Moneyness m')
axes[0].set_ylabel('R² moyen')
axes[0].set_title(f'R² moyen de la régression Δσ ~ ΔlnS\n(fenêtre {ROLL_DEFAULT})')
axes[0].legend()

# Distribution du R² ATM
r2_atm = r2_df[1.00].dropna()
axes[1].hist(r2_atm.values, bins=50, color='steelblue', alpha=0.8,
             density=True, edgecolor='k', lw=0.3)
axes[1].axvline(r2_atm.mean(), color='firebrick', lw=2,
                label=f'Moyenne = {r2_atm.mean():.3f}')
axes[1].set_xlabel('R²')
axes[1].set_title(f'Distribution du R² ATM\nContraste OLS 1 jour vs {ROLL_DEFAULT} glissant')
axes[1].legend()

# Comparaison R² selon la magnitude du mouvement spot
d_vol_atm  = mat_ts.diff().dropna()
d_spot_abs = np.log(spot_ts).diff().dropna().abs()
common3    = d_vol_atm.index.intersection(d_spot_abs.index)

# R² OLS 1 jour (naïf)
r2_daily = []
d_s_vals  = d_spot_abs.loc[common3].values
d_v_vals  = d_vol_atm.loc[common3, 1.00].values
for i in range(len(d_s_vals)):
    r2_daily.append(d_s_vals[i])

# Scatter R² glissant vs |ΔlnS| du dernier jour
d_spot_last = np.log(spot_ts).diff().dropna().abs()
d_spot_aligned = d_spot_last.reindex(r2_df.index, method='nearest').dropna()
r2_atm_aligned = r2_df[1.00].reindex(d_spot_aligned.index).dropna()
common4 = d_spot_aligned.index.intersection(r2_atm_aligned.index)

axes[2].scatter(
    d_spot_aligned.loc[common4].values * 100,
    r2_atm_aligned.loc[common4].values,
    alpha=0.3, s=8, color='steelblue'
)
axes[2].set_xlabel('|Δ ln S| dernier jour (%)')
axes[2].set_ylabel('R² (fenêtre glissante)')
axes[2].set_title('R² glissant indépendant de |ΔS|\n(vs OLS 1 jour qui s\'effondre)')
axes[2].axhline(r2_atm.mean(), ls='--', color='firebrick', lw=1.5,
                label=f'R² moyen = {r2_atm.mean():.3f}')
axes[2].legend()

plt.suptitle(f'Section 3 — Diagnostics de la régression glissante ({TICKER_FOCUS}, {ROLL_DEFAULT})',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 4 — Courbe $\beta(m)$ : signature du régime sticky

In [ ]:
# ============================================================
#  COURBES β(m) THÉORIQUES DES RÉGIMES PURS
# ============================================================
def beta_sticky_strike(m_grid, smile_slope):
    """
    β_SS(m) = -dσ/dm * m  (en coordonnées de variations absolues de vol).
    Quand le spot monte, le smile se déplace vers la droite.
    smile_slope : dσ/dm évalué en chaque point m.
    """
    return -smile_slope * m_grid  # pente locale × moneyness


def beta_sticky_skew(m_grid, atm_vol, skew_slope):
    """
    β_SK(m) = 2 * skew * σ_ATM  (shift uniforme × 2).
    C'est le résultat de Bergomi IV : R=2 → shift ATM = 2 × skew × ΔlnS.
    """
    return 2 * skew_slope * atm_vol * np.ones(len(m_grid))


def beta_sticky_delta(m_grid):
    """β_SD(m) = 0 partout (surface fixe en moneyness)."""
    return np.zeros(len(m_grid))


# ─── Calcul des courbes théoriques à la dernière date ──────
last_date  = beta_df.dropna().index[-1]
smile_last = mat_ts.loc[last_date].values   # vol implicite au dernier jour

# Pente locale du smile : dσ/dm par différences finies
dσ_dm = np.gradient(smile_last, M_GRID)
σ_atm = smile_last[np.argmin(np.abs(M_GRID - 1.))]
skew_atm = dσ_dm[np.argmin(np.abs(M_GRID - 1.))]

β_SS = beta_sticky_strike(M_GRID, dσ_dm)
β_SK = beta_sticky_skew(M_GRID, σ_atm, skew_atm)
β_SD = beta_sticky_delta(M_GRID)

# β observé (dernière date disponible)
β_obs = beta_df.loc[last_date].values
β_se  = se_df.loc[last_date].values

# ─── Figure principale ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gauche : β(m) observé vs régimes purs
axes[0].fill_between(
    M_GRID,
    (β_obs - 2*β_se) * 100,
    (β_obs + 2*β_se) * 100,
    alpha=0.2, color='steelblue', label='IC 95%'
)
axes[0].plot(M_GRID, β_obs * 100, 'steelblue', lw=3, label='β observé', zorder=5)
axes[0].plot(M_GRID, β_SS * 100, 'firebrick', lw=2, ls='--', label='Sticky Strike (théorique)')
axes[0].plot(M_GRID, β_SK * 100, 'forestgreen', lw=2, ls=':', label='Sticky Skew (théorique, R=2)')
axes[0].plot(M_GRID, β_SD * 100, 'orange', lw=2, ls='-.', label='Sticky Delta (β=0)')
axes[0].axhline(0., color='k', lw=0.8)
axes[0].axvline(1., color='gray', lw=0.8, ls='--')
axes[0].set_xlabel('Moneyness m')
axes[0].set_ylabel('β(m) [variation vol / variation lnS]')
axes[0].set_title(f'{TICKER_FOCUS} — Courbe β(m)\n'
                  f'Fenêtre {ROLL_DEFAULT}, dernière date : {pd.Timestamp(last_date).date()}')
axes[0].legend(fontsize=9)

# Droite : position dans le triangle des régimes
# On projette β_obs sur [β_SD, β_SS, β_SK] par OLS
A    = np.column_stack([β_SD, β_SS, β_SK])
y_r  = β_obs
mask = np.isfinite(y_r)
if mask.sum() >= 3:
    coeffs, _, _, _ = lstsq(A[mask], y_r[mask])
    w_sd, w_ss, w_sk = coeffs
    fit  = A @ coeffs
    ss_res = np.sum((y_r[mask] - fit[mask])**2)
    ss_tot = np.sum((y_r[mask] - y_r[mask].mean())**2)
    r2_fit = 1 - ss_res/ss_tot if ss_tot > 0 else 0.
else:
    w_sd, w_ss, w_sk = 0., 0., 0.; r2_fit = 0.

# Normaliser pour affichage
w_sum = abs(w_sd) + abs(w_ss) + abs(w_sk) + 1e-10
w_sd_n, w_ss_n, w_sk_n = w_sd/w_sum, w_ss/w_sum, w_sk/w_sum

labels_r = ['Sticky\nDelta', 'Sticky\nStrike', 'Sticky\nSkew']
vals_r   = [abs(w_sd_n)*100, abs(w_ss_n)*100, abs(w_sk_n)*100]
colors_r = ['orange', 'firebrick', 'forestgreen']

bars = axes[1].bar(labels_r, vals_r, color=colors_r, alpha=0.8,
                   edgecolor='k', lw=0.8, width=0.5)
for bar, val in zip(bars, vals_r):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Poids dans la décomposition (%)')
axes[1].set_title(f'Décomposition du régime\n(R²={r2_fit:.3f})')
axes[1].set_ylim(0, max(vals_r)*1.25)

plt.suptitle(f'Section 4 — Courbe β(m) : signature du régime sticky ({TICKER_FOCUS})',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nDécomposition du régime actuel ({pd.Timestamp(last_date).date()}) :')
print(f'  Sticky Delta  : {abs(w_sd_n)*100:.1f}%')
print(f'  Sticky Strike : {abs(w_ss_n)*100:.1f}%')
print(f'  Sticky Skew   : {abs(w_sk_n)*100:.1f}%')
print(f'  R² de la décomposition : {r2_fit:.4f}')

---
## Section 5 — SSR normalisé $\tilde{\beta}(m)$ : comparaison aux régimes purs

In [ ]:
# ============================================================
#  SSR NORMALISÉ β̃(m) = β(m) / skew_ATM
# ============================================================
def compute_normalized_SSR(beta_df, mat_ts):
    """
    Normalise β(m,t) par le skew ATM pour obtenir β̃(m,t).

    β̃(m) = β(m) / |dσ/dlnK|_{ATM}

    Interprétation :
      β̃ = 0  → Sticky Delta
      β̃ = 1  → Sticky Strike
      β̃ = 2  → Sticky Skew
    """
    m_cols = mat_ts.columns.astype(float)
    atm_idx = np.argmin(np.abs(m_cols - 1.0))

    # Skew ATM : pente locale à m=1, estimée par différences finies
    skew_ts = pd.Series(index=mat_ts.index, dtype=float)
    for date in mat_ts.index:
        smile = mat_ts.loc[date].values
        dσ_dm = np.gradient(smile, m_cols.values)
        # Conversion en skew log-moneyness : dσ/d(lnK) = dσ/dm * m
        skew_ts.loc[date] = dσ_dm[atm_idx] * m_cols[atm_idx]

    # Normalisation
    beta_norm = beta_df.copy()
    skew_aligned = skew_ts.reindex(beta_df.index).dropna()
    common = beta_df.index.intersection(skew_aligned.index)

    for col in beta_df.columns:
        beta_norm.loc[common, col] = (
            beta_df.loc[common, col] / skew_aligned.loc[common].abs()
        )

    return beta_norm, skew_ts


beta_norm, skew_series = compute_normalized_SSR(beta_df, mat_ts)

# ─── Visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# β̃(m) au dernier jour
beta_tilde_last = beta_norm.loc[last_date].values

axes[0].axhspan(0., 1., alpha=0.08, color='orange',     label='Sticky Delta→Strike')
axes[0].axhspan(1., 2., alpha=0.08, color='forestgreen', label='Sticky Strike→Skew')
axes[0].axhline(0., color='orange',     lw=2,   ls='-.', label='β̃=0 (Sticky Delta)')
axes[0].axhline(1., color='firebrick',  lw=2,   ls='--', label='β̃=1 (Sticky Strike)')
axes[0].axhline(2., color='forestgreen',lw=2,   ls=':',  label='β̃=2 (Sticky Skew)')
axes[0].plot(M_GRID, beta_tilde_last, 'steelblue', lw=3, zorder=5, label='β̃ observé')
axes[0].fill_between(M_GRID,
    beta_norm.quantile(0.25).values,
    beta_norm.quantile(0.75).values,
    alpha=0.2, color='steelblue', label='IQR historique'
)
axes[0].axvline(1., color='gray', lw=0.8, ls='--')
axes[0].set_xlabel('Moneyness m')
axes[0].set_ylabel('β̃(m) = β(m) / |skew ATM|')
axes[0].set_title(f'SSR normalisé β̃(m)\nDernier jour : {pd.Timestamp(last_date).date()}')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].set_ylim(-1, 3)

# SSR ATM dans le temps
ssr_atm_ts = beta_norm[1.00].dropna()
axes[1].plot(ssr_atm_ts.index, ssr_atm_ts.values, 'steelblue', lw=1.5)
axes[1].fill_between(ssr_atm_ts.index, 0., 1., alpha=0.08, color='orange')
axes[1].fill_between(ssr_atm_ts.index, 1., 2., alpha=0.08, color='forestgreen')
axes[1].axhline(0., color='orange',     lw=1.5, ls='-.')
axes[1].axhline(1., color='firebrick',  lw=1.5, ls='--')
axes[1].axhline(2., color='forestgreen',lw=1.5, ls=':')
axes[1].set_ylabel('SSR ATM β̃(1.0)')
axes[1].set_title(f'SSR ATM dans le temps\n(fenêtre {ROLL_DEFAULT})')
axes[1].set_ylim(-0.5, 3.)
axes[1].xaxis.set_tick_params(rotation=30)

# Distribution historique du SSR ATM
axes[2].hist(ssr_atm_ts.dropna().values, bins=60,
             color='steelblue', alpha=0.8, density=True, edgecolor='k', lw=0.3,
             orientation='horizontal')
axes[2].axhline(0., color='orange',     lw=2, ls='-.',  label='SD (β̃=0)')
axes[2].axhline(1., color='firebrick',  lw=2, ls='--',  label='SS (β̃=1)')
axes[2].axhline(2., color='forestgreen',lw=2, ls=':',   label='SK (β̃=2)')
axes[2].axhline(ssr_atm_ts.mean(), color='k', lw=2.5,
                label=f'Moy={ssr_atm_ts.mean():.3f}')
axes[2].set_xlabel('Densité')
axes[2].set_ylabel('SSR β̃(1.0)')
axes[2].set_title('Distribution historique du SSR ATM')
axes[2].legend(fontsize=8)
axes[2].set_ylim(-0.5, 3.)

plt.suptitle(f'Section 5 — SSR normalisé β̃(m) ({TICKER_FOCUS}, {ROLL_DEFAULT})',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'SSR ATM moyen : {ssr_atm_ts.mean():.3f}')
print(f'  → 0 = Sticky Delta, 1 = Sticky Strike, 2 = Sticky Skew')
print(f'SSR ATM dernier jour : {beta_norm.loc[last_date, 1.00]:.3f}')

---
## Section 6 — Décomposition polynomiale $(w_0, w_1, w_2)$

On projette la courbe $\beta(m)$ sur trois formes de base :
$$\beta(m) \approx w_0 \cdot \mathbf{1} + w_1 \cdot (m-1) + w_2 \cdot (m-1)^2$$

| Coefficient | Interprétation |
|---|---|
| $w_0$ | Niveau moyen de réponse (sticky skew-like) |
| $w_1$ | Asymétrie de la réponse (biais vers puts ou calls) |
| $w_2$ | Courbure (régime différent puts vs calls) |

In [ ]:
# ============================================================
#  DÉCOMPOSITION POLYNOMIALE DE β(m)
# ============================================================
def polynomial_decompose(beta_df, m_grid, degree=2):
    """
    Projette β(m,t) sur la base polynomiale centrée en m=1.
    Retourne DataFrame (dates × [w0, w1, w2, r2_poly]).
    """
    # Base : [1, (m-1), (m-1)²]
    m_c = m_grid - 1.0
    basis = np.column_stack([m_c**k for k in range(degree+1)])  # (n_m, degree+1)

    results = []
    for date, row in beta_df.iterrows():
        y = row.values.astype(float)
        mask = np.isfinite(y)
        if mask.sum() < degree + 2:
            results.append([np.nan]*(degree+2))
            continue
        try:
            coeffs, _, _, _ = lstsq(basis[mask], y[mask])
            y_fit   = basis @ coeffs
            ss_res  = np.sum((y[mask] - y_fit[mask])**2)
            ss_tot  = np.sum((y[mask] - y[mask].mean())**2)
            r2_poly = 1 - ss_res/ss_tot if ss_tot > 0 else 0.
            results.append(list(coeffs) + [r2_poly])
        except:
            results.append([np.nan]*(degree+2))

    cols = [f'w{k}' for k in range(degree+1)] + ['r2_poly']
    return pd.DataFrame(results, index=beta_df.index, columns=cols)


poly_df = polynomial_decompose(beta_df, M_GRID.astype(float), degree=2)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ─── Top : séries temporelles des coefficients ─────────────
coeff_labels = {
    'w0': ('Niveau $w_0$ — réponse uniforme\n(≈ sticky skew)', 'steelblue'),
    'w1': ('Asymétrie $w_1$ — biais puts/calls\n(≈ mixte SS/SD)', 'firebrick'),
    'w2': ('Courbure $w_2$ — régime différentiel\n(≈ butterfly)', 'forestgreen'),
}

for ax_idx, (col, (label, color)) in enumerate(coeff_labels.items()):
    ts = poly_df[col].dropna()
    axes[0, ax_idx].plot(ts.index, ts.values, color=color, lw=1.5, alpha=0.9)
    axes[0, ax_idx].axhline(ts.mean(), color='k', lw=2, ls='--',
                             label=f'Moyenne = {ts.mean():.4f}')
    axes[0, ax_idx].axhline(0., color='gray', lw=0.8)
    axes[0, ax_idx].set_title(label)
    axes[0, ax_idx].legend(fontsize=9)
    axes[0, ax_idx].xaxis.set_tick_params(rotation=30)

# ─── Bottom : fit au dernier jour + distributions ──────────
m_c = M_GRID.astype(float) - 1.0
basis_last = np.column_stack([m_c**k for k in range(3)])
last_coeffs = poly_df.loc[last_date, ['w0','w1','w2']].values
beta_fitted  = basis_last @ last_coeffs

axes[1, 0].plot(M_GRID, beta_df.loc[last_date].values * 100,
                'steelblue', lw=2.5, label='β observé')
axes[1, 0].plot(M_GRID, beta_fitted * 100,
                'firebrick', lw=2, ls='--', label=f'Fit poly. deg.2 (R²={poly_df.loc[last_date,"r2_poly"]:.3f})')
axes[1, 0].axvline(1., color='gray', lw=0.8, ls='--')
axes[1, 0].axhline(0., color='k', lw=0.7)
axes[1, 0].set_xlabel('Moneyness m')
axes[1, 0].set_ylabel('β(m)')
axes[1, 0].set_title(f'Fit polynomial au {pd.Timestamp(last_date).date()}')
axes[1, 0].legend(fontsize=9)

# Distribution de w0, w1
for ax_idx, (col, color) in enumerate([('w0','steelblue'),('w1','firebrick')], start=1):
    ts = poly_df[col].dropna()
    axes[1, ax_idx].hist(ts.values, bins=50, color=color, alpha=0.8,
                          density=True, edgecolor='k', lw=0.3)
    axes[1, ax_idx].axvline(ts.mean(),   color='k', lw=2,   label=f'μ={ts.mean():.4f}')
    axes[1, ax_idx].axvline(ts.iloc[-1], color='r', lw=2.5, ls='--',
                             label=f'Actuel={ts.iloc[-1]:.4f}')
    axes[1, ax_idx].set_xlabel(col)
    axes[1, ax_idx].set_title(f'Distribution de {col}')
    axes[1, ax_idx].legend(fontsize=9)

plt.suptitle(f'Section 6 — Décomposition polynomiale β(m) = w₀ + w₁(m-1) + w₂(m-1)² ({TICKER_FOCUS})',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Coefficients au {pd.Timestamp(last_date).date()} :')
print(f'  w₀ (niveau)    = {last_coeffs[0]:.5f}')
print(f'  w₁ (asymétrie) = {last_coeffs[1]:.5f}')
print(f'  w₂ (courbure)  = {last_coeffs[2]:.5f}')

---
## Section 7 — Décomposition quotidienne : part spot vs part autonome

Pour chaque jour $t$, on décompose $\Delta\hat{\sigma}^{\text{obs}}(m)$ en deux parts :

$$\underbrace{\Delta\hat{\sigma}^{\text{obs}}(m)}_\text{observé} = \underbrace{\hat{\beta}(m) \cdot \Delta\ln S_t}_\text{part expliquée par le spot} + \underbrace{\hat{\alpha}(m) + \hat{\varepsilon}_t(m)}_\text{part autonome (idiosyncratique)}$$

C'est ce qui intéresse directement le trader pour savoir si la variation du smile d'hier à aujourd'hui est due au mouvement du spot ou à un choc idiosyncratique.

In [ ]:
# ============================================================
#  DÉCOMPOSITION QUOTIDIENNE DU MOUVEMENT DU SMILE
# ============================================================
def decompose_daily_move(mat_ts, spot_ts, beta_df, alpha_df, date_t1=None):
    """
    Pour une date donnée t1, décompose le mouvement du smile
    entre t0 (veille) et t1 (aujourd'hui).

    Retourne :
      delta_obs      : Δσ observé (m,)
      delta_spot     : part expliquée par ΔlnS (m,)
      delta_auto     : part autonome (m,)
      delta_lnS      : rendement spot du jour
      r2_decomp      : R² de la décomposition
    """
    if date_t1 is None:
        date_t1 = mat_ts.index[-1]

    # Trouver t0 (veille de bourse)
    dates_avail = mat_ts.index.tolist()
    idx_t1 = dates_avail.index(date_t1) if date_t1 in dates_avail else -1
    if idx_t1 <= 0:
        return None
    date_t0 = dates_avail[idx_t1 - 1]

    # Variation de vol observée
    delta_obs = (mat_ts.loc[date_t1] - mat_ts.loc[date_t0]).values

    # Rendement spot
    S_t1 = spot_ts.asof(pd.Timestamp(date_t1))
    S_t0 = spot_ts.asof(pd.Timestamp(date_t0))
    if np.isnan(S_t1) or np.isnan(S_t0) or S_t0 <= 0:
        return None
    delta_lnS = np.log(S_t1 / S_t0)

    # β et α au jour t1
    if date_t1 not in beta_df.index:
        date_t1_beta = beta_df.index[beta_df.index <= date_t1][-1]
    else:
        date_t1_beta = date_t1

    beta_t  = beta_df.loc[date_t1_beta].values
    alpha_t = alpha_df.loc[date_t1_beta].values

    # Décomposition
    delta_spot = beta_t * delta_lnS        # part spot
    delta_auto = delta_obs - delta_spot    # part autonome

    # R² de la décomposition
    mask = np.isfinite(delta_obs) & np.isfinite(delta_spot)
    ss_spot = np.sum(delta_spot[mask]**2)
    ss_tot  = np.sum(delta_obs[mask]**2)
    r2_d    = ss_spot / ss_tot if ss_tot > 0 else 0.

    return {
        'date_t0': date_t0, 'date_t1': date_t1,
        'delta_obs':   delta_obs,
        'delta_spot':  delta_spot,
        'delta_auto':  delta_auto,
        'delta_lnS':   delta_lnS,
        'r2_decomp':   r2_d,
        'm_grid':      mat_ts.columns.astype(float).values
    }


# Décomposition du dernier jour
decomp_today = decompose_daily_move(mat_ts, spot_ts, beta_df, alpha_df)

if decomp_today is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    m_d = decomp_today['m_grid']

    # Décomposition graphique
    axes[0].plot(m_d, decomp_today['delta_obs'] * 100,
                 'steelblue', lw=3, label='Δσ observé')
    axes[0].plot(m_d, decomp_today['delta_spot'] * 100,
                 'firebrick', lw=2.5, ls='--', label=f'Part spot (ΔlnS={decomp_today["delta_lnS"]*100:.2f}%)')
    axes[0].fill_between(m_d,
        decomp_today['delta_spot'] * 100,
        decomp_today['delta_obs']  * 100,
        alpha=0.25, color='forestgreen', label='Part autonome')
    axes[0].axhline(0., color='k', lw=0.8)
    axes[0].axvline(1., color='gray', lw=0.8, ls='--')
    axes[0].set_xlabel('Moneyness m')
    axes[0].set_ylabel('Δσ (vol pts, %)')
    axes[0].set_title(f'Décomposition {decomp_today["date_t0"]} → {decomp_today["date_t1"]}\n'
                      f'R²={decomp_today["r2_decomp"]:.3f}')
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Variation de vol (vol pts × 100)')

    # Part spot vs part autonome par point
    width  = 0.03
    axes[1].bar(m_d - width/2, decomp_today['delta_spot'] * 100,
                width, color='firebrick', alpha=0.8, label='Part spot')
    axes[1].bar(m_d + width/2, decomp_today['delta_auto'] * 100,
                width, color='forestgreen', alpha=0.8, label='Part autonome')
    axes[1].axhline(0., color='k', lw=0.8)
    axes[1].axvline(1., color='gray', lw=0.8, ls='--')
    axes[1].set_xlabel('Moneyness m')
    axes[1].set_ylabel('Contribution (vol pts × 100)')
    axes[1].set_title('Part spot vs Part autonome\npar point de moneyness')
    axes[1].legend(fontsize=9)

    # Camembert de la variance expliquée
    mask_d = np.isfinite(decomp_today['delta_obs'])
    var_spot = np.sum(decomp_today['delta_spot'][mask_d]**2)
    var_auto = np.sum(decomp_today['delta_auto'][mask_d]**2)
    var_tot  = np.sum(decomp_today['delta_obs'][mask_d]**2)

    pct_spot = var_spot / var_tot * 100 if var_tot > 0 else 0
    pct_auto = var_auto / var_tot * 100 if var_tot > 0 else 0

    axes[2].pie(
        [max(pct_spot,0), max(pct_auto,0)],
        labels=[f'Part spot\n{pct_spot:.1f}%', f'Part autonome\n{pct_auto:.1f}%'],
        colors=['firebrick', 'forestgreen'],
        startangle=90, autopct='%1.1f%%', pctdistance=0.75,
        wedgeprops=dict(alpha=0.85, edgecolor='k', lw=0.8)
    )
    axes[2].set_title(f'Décomposition de la variance du Δσ\n'
                      f'(ΔlnS = {decomp_today["delta_lnS"]*100:.2f}%)')

    plt.suptitle(f'Section 7 — Décomposition quotidienne du mouvement du smile ({TICKER_FOCUS})',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'\nDécomposition du {decomp_today["date_t0"]} → {decomp_today["date_t1"]} :')
    print(f'  ΔlnS = {decomp_today["delta_lnS"]*100:.3f}%')
    print(f'  Part expliquée par le spot : {pct_spot:.1f}%')
    print(f'  Part autonome (idiosyncratique) : {pct_auto:.1f}%')

---
## Section 8 — Évolution temporelle du régime

In [ ]:
# ============================================================
#  ÉVOLUTION TEMPORELLE : RÉGIME STICKY AU COURS DU TEMPS
# ============================================================
# On trace β̃(m=1) dans le temps avec coloration par régime

ssr_ts = beta_norm[1.00].dropna()

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# ─── Haut : SSR ATM + zones de régime ─────────────────────
ax = axes[0]

# Zones colorées
ax.axhspan(-99, 0.5, alpha=0.07, color='orange',     label='≈ Sticky Delta')
ax.axhspan(0.5, 1.5, alpha=0.07, color='firebrick',  label='≈ Sticky Strike')
ax.axhspan(1.5, 99,  alpha=0.07, color='forestgreen', label='≈ Sticky Skew')

ax.plot(ssr_ts.index, ssr_ts.values, 'steelblue', lw=1.8, alpha=0.9)

# Colorier la courbe selon le régime
for i in range(len(ssr_ts)-1):
    v = ssr_ts.iloc[i]
    col = 'orange' if v < 0.5 else ('firebrick' if v < 1.5 else 'forestgreen')
    ax.plot(ssr_ts.index[i:i+2], ssr_ts.values[i:i+2], color=col, lw=3, alpha=0.6)

ax.axhline(0., color='orange',     lw=1.5, ls='-.',  alpha=0.8)
ax.axhline(1., color='firebrick',  lw=1.5, ls='--',  alpha=0.8)
ax.axhline(2., color='forestgreen',lw=1.5, ls=':',   alpha=0.8)

# Moyenne glissante 3M
ssr_roll = ssr_ts.rolling(63).mean()
ax.plot(ssr_roll.index, ssr_roll.values, 'k', lw=2.5, label='Moyenne 3M', zorder=5)

ax.set_ylabel('SSR β̃(m=1)')
ax.set_title(f'{TICKER_FOCUS} — Évolution du SSR ATM ({ROLL_DEFAULT})')
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(-0.5, 3.)

# ─── Milieu : rendements spot ──────────────────────────────
spot_ret = np.log(spot_ts).diff().dropna()
spot_ret_aligned = spot_ret.reindex(ssr_ts.index, method='nearest')

axes[1].bar(spot_ret_aligned.index, spot_ret_aligned.values * 100,
            color=np.where(spot_ret_aligned.values > 0, 'forestgreen', 'firebrick'),
            alpha=0.7, width=1.)
axes[1].axhline(0., color='k', lw=0.8)
axes[1].set_ylabel('Δ ln S (%)')
axes[1].set_title('Rendements journaliers du sous-jacent')

# ─── Bas : w₀, w₁ (coefficients polynomiaux) ──────────────
poly_aligned = poly_df.reindex(ssr_ts.index)
axes[2].plot(poly_aligned.index, poly_aligned['w0'].values,
             'steelblue', lw=1.5, label='w₀ (niveau)')
axes[2].plot(poly_aligned.index, poly_aligned['w1'].values,
             'firebrick', lw=1.5, ls='--', label='w₁ (asymétrie)')
axes[2].plot(poly_aligned.index, poly_aligned['w2'].values,
             'forestgreen', lw=1.5, ls=':', label='w₂ (courbure)')
axes[2].axhline(0., color='k', lw=0.8)
axes[2].set_ylabel('Coefficients polynomiaux')
axes[2].set_title('Évolution des coefficients w₀, w₁, w₂')
axes[2].legend(fontsize=9)
axes[2].xaxis.set_tick_params(rotation=30)

plt.suptitle(f'Section 8 — Évolution temporelle du régime sticky ({TICKER_FOCUS})',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  COMPARAISON DES FENÊTRES : 1M vs 3M vs 6M
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# SSR ATM pour les 3 fenêtres
for win_label, color in [('1M','steelblue'),('3M','firebrick'),('6M','forestgreen')]:
    beta_w = BETA_RESULTS[win_label]['beta']
    _, skew_w = compute_normalized_SSR(beta_w, mat_ts)
    beta_norm_w = beta_w.copy()
    sk_al = skew_w.reindex(beta_w.index).dropna()
    for col in beta_w.columns:
        beta_norm_w.loc[sk_al.index, col] = beta_w.loc[sk_al.index, col] / sk_al.abs()
    ssr_w = beta_norm_w[1.00].dropna()
    axes[0].plot(ssr_w.index, ssr_w.rolling(21).mean().values,
                 color=color, lw=2, label=f'Fenêtre {win_label}')

axes[0].axhline(1., color='k', lw=1.5, ls='--', alpha=0.5)
axes[0].axhline(2., color='k', lw=1.5, ls=':', alpha=0.5)
axes[0].set_ylabel('SSR β̃(m=1) — moy. 1M')
axes[0].set_title('SSR ATM : sensibilité à la fenêtre')
axes[0].legend(fontsize=9)
axes[0].set_ylim(-0.5, 3.)
axes[0].xaxis.set_tick_params(rotation=30)

# β(m) à la dernière date — les 3 fenêtres
for win_label, color in [('1M','steelblue'),('3M','firebrick'),('6M','forestgreen')]:
    beta_w = BETA_RESULTS[win_label]['beta']
    last_d = beta_w.dropna().index[-1]
    axes[1].plot(M_GRID, beta_w.loc[last_d].values * 100,
                 color=color, lw=2, label=f'{win_label} (au {pd.Timestamp(last_d).date()})')

axes[1].axhline(0., color='k', lw=0.8)
axes[1].axvline(1., color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('Moneyness m')
axes[1].set_ylabel('β(m)')
axes[1].set_title('Courbe β(m) selon la fenêtre\n(dernière date disponible)')
axes[1].legend(fontsize=9)

plt.suptitle('Robustesse : impact de la fenêtre d\'estimation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 9 — Comparaison multi-ticker & multi-maturité

In [ ]:
# ============================================================
#  ANALYSE MULTI-TICKER : β(m) et SSR pour tous les tickers
# ============================================================
def compute_ticker_regime(ticker, tau_target=3/12., window=63):
    """Calcule le régime sticky pour un ticker donné."""
    if ticker not in SURFACES:
        return None
    surf = SURFACES[ticker]
    result = get_surface_matrix(surf, tau_target)
    if result is None:
        return None
    mat_t, tau_a = result
    spot_t = get_spot_series(ticker)

    beta_t, alpha_t, r2_t, se_t = compute_rolling_beta(mat_t, spot_t, window)
    beta_norm_t, skew_t = compute_normalized_SSR(beta_t, mat_t)
    poly_t = polynomial_decompose(beta_t, M_GRID.astype(float))

    last_d = beta_t.dropna().index[-1]

    return {
        'tau': tau_a,
        'beta_last':  beta_t.loc[last_d].values,
        'beta_norm_last': beta_norm_t.loc[last_d].values if last_d in beta_norm_t.index else None,
        'ssr_atm':    float(beta_norm_t[1.00].dropna().iloc[-1]) if 1.00 in beta_norm_t.columns else np.nan,
        'ssr_atm_mean': float(beta_norm_t[1.00].dropna().mean()) if 1.00 in beta_norm_t.columns else np.nan,
        'w0': float(poly_t['w0'].dropna().iloc[-1]),
        'w1': float(poly_t['w1'].dropna().iloc[-1]),
        'r2_mean': float(r2_t.mean().mean()),
        'last_date': last_d,
    }


MULTI_TICKER = {}
for ticker in TICKERS:
    print(f'Calcul régime {ticker}...')
    res = compute_ticker_regime(ticker, TAU_FOCUS)
    if res is not None:
        MULTI_TICKER[ticker] = res
        print(f'  SSR ATM = {res["ssr_atm"]:.3f}, R² moy = {res["r2_mean"]:.3f}')

In [ ]:
# ============================================================
#  FIGURE COMPARATIVE MULTI-TICKER
# ============================================================
tickers_ok = list(MULTI_TICKER.keys())
n_tickers  = len(tickers_ok)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors_t = plt.cm.tab10(np.linspace(0, 1, n_tickers))

# ─── β(m) pour tous les tickers ─────────────────────────────
for ticker, color in zip(tickers_ok, colors_t):
    res = MULTI_TICKER[ticker]
    axes[0].plot(M_GRID, res['beta_last'] * 100, color=color, lw=2, label=ticker)

axes[0].axhline(0., color='k', lw=0.8)
axes[0].axvline(1., color='gray', lw=0.8, ls='--')
axes[0].set_xlabel('Moneyness m')
axes[0].set_ylabel('β(m) (vol pts / Δ ln S)')
axes[0].set_title(f'Courbe β(m) — tous tickers\nτ ≈ {TAU_FOCUS*12:.0f}M')
axes[0].legend(fontsize=9)

# ─── SSR ATM : actuel vs moyenne historique ──────────────────
x_pos = np.arange(n_tickers)
ssr_now  = [MULTI_TICKER[t]['ssr_atm']      for t in tickers_ok]
ssr_hist = [MULTI_TICKER[t]['ssr_atm_mean'] for t in tickers_ok]

axes[1].bar(x_pos - 0.2, ssr_hist, 0.35, alpha=0.5, color=colors_t,
            edgecolor='k', lw=0.8, label='Historique')
axes[1].bar(x_pos + 0.2, ssr_now, 0.35,  alpha=0.9, color=colors_t,
            edgecolor='k', lw=0.8, label='Actuel')
axes[1].axhline(1., color='firebrick',  lw=1.5, ls='--', label='SS (β̃=1)')
axes[1].axhline(2., color='forestgreen',lw=1.5, ls=':',  label='SK (β̃=2)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(tickers_ok, fontsize=10)
axes[1].set_ylabel('SSR β̃(m=1)')
axes[1].set_title('SSR ATM : actuel vs historique')
axes[1].legend(fontsize=8)
axes[1].set_ylim(-0.5, 3.5)

# ─── Scatter w₀ vs w₁ (espace des régimes) ──────────────────
for ticker, color in zip(tickers_ok, colors_t):
    res = MULTI_TICKER[ticker]
    axes[2].scatter(res['w1'], res['w0'], s=200, color=color,
                   edgecolors='k', lw=1.5, zorder=5)
    axes[2].annotate(ticker, (res['w1'], res['w0']),
                    textcoords='offset points', xytext=(8, 4), fontsize=10)

axes[2].axhline(0., color='k', lw=0.8)
axes[2].axvline(0., color='k', lw=0.8)
axes[2].set_xlabel('w₁ (asymétrie puts/calls)')
axes[2].set_ylabel('w₀ (niveau de réponse)')
axes[2].set_title('Espace des régimes (w₀, w₁)\nPosition actuelle de chaque ticker')

# Ajouter des annotations de régime
axes[2].text(0.02, 0.02, 'Sticky Delta\n(w₀≈0)', transform=axes[2].transAxes,
             color='orange', fontsize=9, alpha=0.7)
axes[2].text(0.02, 0.85, 'Sticky Skew\n(w₀>0)', transform=axes[2].transAxes,
             color='forestgreen', fontsize=9, alpha=0.7)

plt.suptitle(f'Section 9 — Comparaison multi-ticker (τ ≈ {TAU_FOCUS*12:.0f}M, fenêtre {ROLL_DEFAULT})',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  ANALYSE MULTI-MATURITÉ — β(m) par τ pour un ticker
# ============================================================
tau_labels = {1/12.:'1M', 2/12.:'2M', 3/12.:'3M', 6/12.:'6M', 9/12.:'9M', 12/12.:'12M'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors_tau = plt.cm.viridis(np.linspace(0.1, 0.9, len(TAU_GRID)))

ssr_by_tau  = []
beta_by_tau = []

for tau_t, color in zip(TAU_GRID, colors_tau):
    result = get_surface_matrix(surf_focus, tau_t)
    if result is None:
        continue
    mat_tau, tau_a = result
    beta_tau, alpha_tau, _, _ = compute_rolling_beta(mat_tau, spot_ts,
                                                      ROLL_WINDOWS[ROLL_DEFAULT])
    beta_n_tau, sk_tau = compute_normalized_SSR(beta_tau, mat_tau)

    last_d = beta_tau.dropna().index[-1]

    # β(m)
    axes[0].plot(M_GRID, beta_tau.loc[last_d].values * 100,
                 color=color, lw=2, label=tau_labels.get(tau_t, f'{tau_t*12:.0f}M'))

    # SSR ATM
    ssr_atm_v = float(beta_n_tau[1.00].dropna().iloc[-1]) if 1.00 in beta_n_tau.columns else np.nan
    ssr_by_tau.append((tau_a, ssr_atm_v))
    beta_by_tau.append((tau_a, beta_tau.loc[last_d].values))

axes[0].axhline(0., color='k', lw=0.8)
axes[0].axvline(1., color='gray', lw=0.8, ls='--')
axes[0].set_xlabel('Moneyness m')
axes[0].set_ylabel('β(m)')
axes[0].set_title(f'{TICKER_FOCUS} — β(m) par maturité')
axes[0].legend(fontsize=8)

# SSR ATM par maturité
tau_arr  = [x[0] for x in ssr_by_tau]
ssr_arr  = [x[1] for x in ssr_by_tau]
axes[1].plot([t*12 for t in tau_arr], ssr_arr, 'o-', color='steelblue', lw=2, ms=8)
axes[1].axhline(1., color='firebrick',  lw=1.5, ls='--', label='Sticky Strike')
axes[1].axhline(2., color='forestgreen',lw=1.5, ls=':',  label='Sticky Skew')
axes[1].set_xlabel('Maturité (mois)')
axes[1].set_ylabel('SSR β̃(m=1)')
axes[1].set_title('SSR ATM par maturité')
axes[1].legend(fontsize=9)
axes[1].set_ylim(-0.5, 3.)

# Heatmap β(m, τ)
if len(beta_by_tau) > 0:
    tau_mat = np.array([x[0] for x in beta_by_tau])
    beta_mat = np.array([x[1] for x in beta_by_tau])
    im = axes[2].imshow(
        beta_mat * 100, aspect='auto', origin='lower',
        cmap='RdBu_r',
        extent=[M_GRID[0], M_GRID[-1], tau_mat[0]*12, tau_mat[-1]*12]
    )
    plt.colorbar(im, ax=axes[2], label='β(m,τ)')
    axes[2].set_xlabel('Moneyness m')
    axes[2].set_ylabel('Maturité (mois)')
    axes[2].set_title('Heatmap β(m, τ)\n(rouge = réponse negative, bleu = positive)')

plt.suptitle(f'Section 9b — Analyse multi-maturité ({TICKER_FOCUS})', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 10 — Interface trader : vue synthétique journalière

In [ ]:
# ============================================================
#  INTERFACE TRADER — VUE SYNTHÉTIQUE
#  Sélectionner un ticker et une maturité pour voir le rapport complet
# ============================================================

# ─── PARAMÈTRES À MODIFIER ─────────────────────────────────
TICKER_TRADE = 'AAPL'   # ticker à analyser
TAU_TRADE    = 3/12.    # maturité en années (1/12.=1M, 3/12.=3M, etc.)
WINDOW_TRADE = '3M'     # fenêtre de régression
# ────────────────────────────────────────────────────────────

def trader_report(ticker, tau_target, window_label='3M'):
    """Génère le rapport complet pour le trader."""
    if ticker not in SURFACES:
        print(f'Ticker {ticker} non disponible.')
        return

    surf  = SURFACES[ticker]
    result = get_surface_matrix(surf, tau_target)
    if result is None:
        print(f'Maturité {tau_target*12:.0f}M non disponible pour {ticker}.')
        return
    mat_t, tau_a = result
    spot_t = get_spot_series(ticker)

    window = ROLL_WINDOWS[window_label]
    beta_t, alpha_t, r2_t, se_t = compute_rolling_beta(mat_t, spot_t, window)
    beta_norm_t, skew_t = compute_normalized_SSR(beta_t, mat_t)
    poly_t = polynomial_decompose(beta_t, M_GRID.astype(float))

    last_d = beta_t.dropna().index[-1]
    decomp = decompose_daily_move(mat_t, spot_t, beta_t, alpha_t, last_d)

    # ─── Figure principale ─────────────────────────────────
    fig = plt.figure(figsize=(20, 16))
    gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # ── Titre et méta-données ──
    S_today = spot_t.iloc[-1]
    dS_pct  = (np.log(spot_t.iloc[-1]/spot_t.iloc[-2])*100
               if len(spot_t) > 1 else 0.)
    atm_vol_today = mat_t.iloc[-1][1.00] if 1.00 in mat_t.columns else np.nan
    skew_today = mat_t.iloc[-1].get(0.90, np.nan) - mat_t.iloc[-1].get(1.10, np.nan)
    ssr_today  = float(beta_norm_t[1.00].dropna().iloc[-1])
    ssr_hist   = float(beta_norm_t[1.00].dropna().mean())

    fig.text(0.02, 0.98, f'STICKY REGIME REPORT — {ticker}',
             fontsize=16, fontweight='bold', va='top')
    fig.text(0.02, 0.955,
             f'Maturité : {tau_a*12:.0f}M  |  Fenêtre : {window_label}  |  '
             f'Date : {pd.Timestamp(last_d).date()}',
             fontsize=12, va='top', color='gray')

    # ── Métriques clés (KPIs) ──
    kpis = [
        (f'Spot: {S_today:.1f}', f'({dS_pct:+.2f}%)',
         'forestgreen' if dS_pct >= 0 else 'firebrick'),
        (f'Vol ATM: {atm_vol_today*100:.1f}%', f'(τ={tau_a*12:.0f}M)', 'steelblue'),
        (f'Skew 90-110: {skew_today*100:.2f}%', '', 'darkorange'),
        (f'SSR actuel: {ssr_today:.3f}', f'(hist: {ssr_hist:.3f})',
         'forestgreen' if ssr_today > 1.5 else ('firebrick' if ssr_today < 0.5 else 'steelblue')),
    ]
    for k_idx, (main, sub, col) in enumerate(kpis):
        x_pos_kpi = 0.02 + k_idx * 0.24
        ax_kpi = fig.add_axes([x_pos_kpi, 0.87, 0.22, 0.06])
        ax_kpi.text(0.5, 0.7, main, ha='center', va='center',
                    fontsize=14, fontweight='bold', color=col,
                    transform=ax_kpi.transAxes)
        ax_kpi.text(0.5, 0.2, sub, ha='center', va='center',
                    fontsize=10, color='gray', transform=ax_kpi.transAxes)
        ax_kpi.set_xlim(0,1); ax_kpi.set_ylim(0,1)
        ax_kpi.axis('off')
        rect = plt.Rectangle((0,0),1,1, fill=False, edgecolor=col, lw=2,
                              transform=ax_kpi.transAxes)
        ax_kpi.add_patch(rect)

    # ── Plot 1 : smile t0 vs t1 ──
    ax1 = fig.add_subplot(gs[0, 0])
    if decomp is not None:
        smile_t0 = mat_t.iloc[-2].values * 100
        smile_t1 = mat_t.iloc[-1].values * 100
        ax1.plot(M_GRID, smile_t0, 'steelblue', lw=2, ls='--',
                 label=f't₀ ({decomp["date_t0"]})')
        ax1.plot(M_GRID, smile_t1, 'firebrick', lw=2.5,
                 label=f't₁ ({decomp["date_t1"]})')
    ax1.axvline(1., color='gray', lw=0.8, ls='--')
    ax1.set_title('Smile t₀ vs t₁', fontsize=10)
    ax1.set_xlabel('m'); ax1.set_ylabel('Vol (%)')
    ax1.legend(fontsize=8)

    # ── Plot 2 : Δσ décomposé ──
    ax2 = fig.add_subplot(gs[0, 1])
    if decomp is not None:
        ax2.plot(M_GRID, decomp['delta_obs'] * 100, 'steelblue', lw=2.5, label='Δσ observé')
        ax2.plot(M_GRID, decomp['delta_spot']* 100, 'firebrick', lw=2, ls='--',
                 label=f'Part spot (ΔlnS={decomp["delta_lnS"]*100:.2f}%)')
        ax2.fill_between(M_GRID, decomp['delta_spot']*100, decomp['delta_obs']*100,
                         alpha=0.3, color='forestgreen', label='Part autonome')
    ax2.axhline(0., color='k', lw=0.7)
    ax2.axvline(1., color='gray', lw=0.8, ls='--')
    ax2.set_title('Δσ : spot vs autonome', fontsize=10)
    ax2.set_xlabel('m'); ax2.legend(fontsize=7)

    # ── Plot 3 : β(m) + régimes purs ──
    ax3 = fig.add_subplot(gs[0, 2])
    beta_last = beta_t.loc[last_d].values
    se_last   = se_t.loc[last_d].values
    ax3.fill_between(M_GRID, (beta_last-2*se_last)*100, (beta_last+2*se_last)*100,
                     alpha=0.2, color='steelblue')
    ax3.plot(M_GRID, beta_last*100, 'steelblue', lw=3, label='β observé')
    smile_last_v = mat_t.loc[last_d].values
    dσ_dm_v = np.gradient(smile_last_v, M_GRID)
    ax3.plot(M_GRID, beta_sticky_strike(M_GRID, dσ_dm_v)*100,
             'firebrick', lw=1.5, ls='--', label='SS théorique')
    ax3.axhline(0., color='k', lw=0.7)
    ax3.axvline(1., color='gray', lw=0.8, ls='--')
    ax3.set_title('β(m) — régime actuel', fontsize=10)
    ax3.set_xlabel('m'); ax3.legend(fontsize=7)

    # ── Plot 4 : SSR normalisé β̃(m) ──
    ax4 = fig.add_subplot(gs[0, 3])
    btn_last = beta_norm_t.loc[last_d].values if last_d in beta_norm_t.index else np.full(len(M_GRID), np.nan)
    ax4.axhspan(-99, 0.5, alpha=0.07, color='orange')
    ax4.axhspan(0.5, 1.5, alpha=0.07, color='firebrick')
    ax4.axhspan(1.5, 99,  alpha=0.07, color='forestgreen')
    ax4.plot(M_GRID, btn_last, 'steelblue', lw=3)
    ax4.axhline(0., color='orange',     lw=1.5, ls='-.',  alpha=0.8)
    ax4.axhline(1., color='firebrick',  lw=1.5, ls='--',  alpha=0.8)
    ax4.axhline(2., color='forestgreen',lw=1.5, ls=':',   alpha=0.8)
    ax4.fill_between(M_GRID,
        beta_norm_t.quantile(0.25).values,
        beta_norm_t.quantile(0.75).values,
        alpha=0.15, color='steelblue')
    ax4.set_ylim(-1, 3)
    ax4.axvline(1., color='gray', lw=0.8, ls='--')
    ax4.set_title('SSR β̃(m)', fontsize=10)
    ax4.set_xlabel('m')

    # ── Plot 5 : SSR ATM historique ──
    ax5 = fig.add_subplot(gs[1, :2])
    ssr_ts_t = beta_norm_t[1.00].dropna()
    for i in range(len(ssr_ts_t)-1):
        v = ssr_ts_t.iloc[i]
        col = 'orange' if v < 0.5 else ('firebrick' if v < 1.5 else 'forestgreen')
        ax5.plot(ssr_ts_t.index[i:i+2], ssr_ts_t.values[i:i+2], color=col, lw=2.5, alpha=0.7)
    ax5.plot(ssr_ts_t.rolling(63).mean().index,
             ssr_ts_t.rolling(63).mean().values, 'k', lw=2.5, label='Moy. 3M')
    ax5.axhline(1., 'firebrick', lw=1.5, ls='--')
    ax5.axhline(2., 'forestgreen', lw=1.5, ls=':')
    ax5.set_ylabel('SSR β̃(1.0)')
    ax5.set_title(f'Historique SSR ATM ({window_label})', fontsize=10)
    ax5.legend(fontsize=8)
    ax5.set_ylim(-0.5, 3.)
    ax5.xaxis.set_tick_params(rotation=30)

    # ── Plot 6 : Poids des régimes (barres) ──
    ax6 = fig.add_subplot(gs[1, 2])
    regime_labels = ['Sticky\nDelta', 'Sticky\nStrike', 'Sticky\nSkew']
    regime_vals   = [max(1.0 - abs(ssr_today), 0),
                     max(1 - abs(ssr_today-1.), 0),
                     max(ssr_today - 1., 0)]
    s_rv = sum(regime_vals) + 1e-10
    regime_pct = [v/s_rv*100 for v in regime_vals]
    bars_r = ax6.bar(regime_labels, regime_pct,
                     color=['orange','firebrick','forestgreen'], alpha=0.85,
                     edgecolor='k', lw=0.8)
    for bar, val in zip(bars_r, regime_pct):
        ax6.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
    ax6.set_title(f'Régime actuel\n(SSR={ssr_today:.3f})', fontsize=10)
    ax6.set_ylim(0, 110)

    # ── Plot 7 : Évolution w₀, w₁ ──
    ax7 = fig.add_subplot(gs[1, 3])
    ax7.plot(poly_t['w0'].dropna().index, poly_t['w0'].dropna().values,
             'steelblue', lw=1.5, label='w₀ (niveau)')
    ax7.plot(poly_t['w1'].dropna().index, poly_t['w1'].dropna().values,
             'firebrick', lw=1.5, ls='--', label='w₁ (asymétrie)')
    ax7.axhline(0., color='k', lw=0.7)
    ax7.set_title('Coefficients poly. w₀, w₁', fontsize=10)
    ax7.legend(fontsize=8)
    ax7.xaxis.set_tick_params(rotation=30)

    # ── Plot 8 : Vol ATM + Spot ──
    ax8  = fig.add_subplot(gs[2, :2])
    ax8b = ax8.twinx()
    spot_plot = spot_t.reindex(mat_t.index, method='nearest')
    ax8.plot(mat_t.index,  mat_t[1.00].values*100,
             'steelblue', lw=1.5, alpha=0.9, label='Vol ATM')
    ax8b.plot(spot_plot.index, spot_plot.values,
              'firebrick', lw=1.2, alpha=0.7, label='Spot')
    ax8.set_ylabel('Vol ATM (%)', color='steelblue')
    ax8b.set_ylabel('Spot', color='firebrick')
    ax8.set_title('Vol ATM & Spot dans le temps', fontsize=10)
    ax8.xaxis.set_tick_params(rotation=30)

    # ── Plot 9 : R² de la régression ──
    ax9 = fig.add_subplot(gs[2, 2])
    r2_atm_t = r2_t[1.00].dropna()
    ax9.plot(r2_atm_t.index, r2_atm_t.rolling(21).mean().values,
             'steelblue', lw=1.5)
    ax9.fill_between(r2_atm_t.index,
        r2_t.min(axis=1).rolling(21).mean().values,
        r2_t.max(axis=1).rolling(21).mean().values,
        alpha=0.2, color='steelblue', label='Min-Max (m)')
    ax9.axhline(r2_atm_t.mean(), color='k', lw=1.5, ls='--',
                label=f'R² moy = {r2_atm_t.mean():.3f}')
    ax9.set_title('R² de la régression Δσ~ΔlnS', fontsize=10)
    ax9.legend(fontsize=8)
    ax9.set_ylim(0, 1)
    ax9.xaxis.set_tick_params(rotation=30)

    # ── Plot 10 : Heatmap β(m) dans le temps ──
    ax10 = fig.add_subplot(gs[2, 3])
    beta_heat = beta_norm_t.dropna(how='all')
    if len(beta_heat) > 5:
        im = ax10.imshow(
            beta_heat.values.T, aspect='auto', origin='lower',
            cmap='RdBu_r', vmin=0, vmax=2.5
        )
        n_dates_h = len(beta_heat)
        ax10.set_xticks(np.linspace(0, n_dates_h-1, 5).astype(int))
        ax10.set_xticklabels(
            [pd.Timestamp(beta_heat.index[i]).strftime('%m/%y')
             for i in np.linspace(0, n_dates_h-1, 5).astype(int)],
            fontsize=7, rotation=30
        )
        ax10.set_yticks(range(len(M_GRID)))
        ax10.set_yticklabels([f'{m:.0%}' for m in M_GRID], fontsize=7)
        plt.colorbar(im, ax=ax10, shrink=0.8, label='β̃(m)')
    ax10.set_title('Heatmap β̃(m,t)', fontsize=10)

    plt.suptitle('', fontsize=1)  # évite le chevauchement
    plt.savefig(f'sticky_report_{ticker}_{tau_a*12:.0f}M.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ─── Résumé textuel ─────────────────────────────────────
    print('\n' + '='*65)
    print(f'  STICKY REGIME REPORT — {ticker} | τ={tau_a*12:.0f}M | {pd.Timestamp(last_d).date()}')
    print('='*65)
    print(f'  Spot : {S_today:.2f}  ({dS_pct:+.3f}%)')
    print(f'  Vol ATM : {atm_vol_today*100:.2f}%')
    print(f'  Skew 90-110 : {skew_today*100:.3f} vol pts')
    print()
    print(f'  SSR actuel β̃(ATM) = {ssr_today:.4f}')
    print(f'  SSR historique moy = {ssr_hist:.4f}')
    regime_name = ('STICKY DELTA' if ssr_today < 0.5 else
                   'STICKY STRIKE' if ssr_today < 1.5 else 'STICKY SKEW')
    print(f'  → Régime dominant : {regime_name}')
    print()
    if decomp is not None:
        pct_s  = decomp['r2_decomp'] * 100
        pct_au = (1 - decomp['r2_decomp']) * 100
        print(f'  Décomposition du mouvement d\'hier ({decomp["delta_lnS"]*100:.2f}% sur S) :')
        print(f'    Part expliquée par le spot  : {pct_s:.1f}%')
        print(f'    Part autonome (idiosync.)   : {pct_au:.1f}%')
    print('='*65)


# Lancer le rapport
trader_report(TICKER_TRADE, TAU_TRADE, WINDOW_TRADE)

---
## Section 11 — Validation croisée & robustesse

In [ ]:
# ============================================================
#  VALIDATION : AUTOCORRÉLATION DU RÉGIME ET PERSISTANCE
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ─── ACF du SSR ATM ─────────────────────────────────────────
ssr_atm_clean = ssr_ts.dropna()
try:
    acf_vals = acf(ssr_atm_clean, nlags=60, fft=True)
    axes[0, 0].bar(range(len(acf_vals)), acf_vals, color='steelblue', alpha=0.7, width=0.8)
    axes[0, 0].axhline(0., color='k', lw=0.7)
    ci = 1.96 / np.sqrt(len(ssr_atm_clean))
    axes[0, 0].axhline( ci, color='firebrick', ls='--', lw=1.5, label='IC 95%')
    axes[0, 0].axhline(-ci, color='firebrick', ls='--', lw=1.5)
    axes[0, 0].set_xlabel('Lag (jours)')
    axes[0, 0].set_title('ACF du SSR ATM\n(persistance du régime)')
    axes[0, 0].legend()
except Exception as e:
    print(f'Erreur ACF : {e}')

# ─── Prédiction du régime du lendemain ──────────────────────
# On teste si le SSR d'aujourd'hui prédit celui de demain
ssr_lag = ssr_atm_clean.shift(1).dropna()
ssr_now = ssr_atm_clean.reindex(ssr_lag.index).dropna()
common  = ssr_lag.index.intersection(ssr_now.index)

if len(common) > 30:
    slope_p, intercept_p, r_p, _, _ = linregress(
        ssr_lag.loc[common].values, ssr_now.loc[common].values
    )
    x_r = np.linspace(ssr_lag.min(), ssr_lag.max(), 100)
    axes[0, 1].scatter(ssr_lag.loc[common].values, ssr_now.loc[common].values,
                       alpha=0.3, s=8, color='steelblue')
    axes[0, 1].plot(x_r, slope_p*x_r+intercept_p, 'firebrick', lw=2.5,
                    label=f'β={slope_p:.3f}, R²={r_p**2:.3f}')
    axes[0, 1].set_xlabel('SSR(t-1)')
    axes[0, 1].set_ylabel('SSR(t)')
    axes[0, 1].set_title('Prédictabilité du SSR\n(SSR(t) vs SSR(t-1))')
    axes[0, 1].legend()

# ─── Stabilité de β(m) : variance de la courbe dans le temps ─
beta_std = beta_df.std()
beta_mean = beta_df.mean()
cv = (beta_std / (beta_mean.abs() + 1e-6)).abs()

axes[0, 2].bar(M_GRID, beta_std.values * 100, width=0.04,
               color='steelblue', alpha=0.8, edgecolor='k', lw=0.5)
axes[0, 2].set_xlabel('Moneyness m')
axes[0, 2].set_ylabel('Std de β(m) dans le temps')
axes[0, 2].set_title('Stabilité de β(m)\n(std temporelle — faible = stable)')

# ─── Comparaison R² par maturité ────────────────────────────
r2_by_mat = []
for tau_t in TAU_GRID:
    result = get_surface_matrix(surf_focus, tau_t)
    if result is None: continue
    mat_tau, _ = result
    beta_tau, _, r2_tau, _ = compute_rolling_beta(mat_tau, spot_ts, ROLL_WINDOWS[ROLL_DEFAULT])
    r2_by_mat.append((tau_t, r2_tau.mean().mean()))

if r2_by_mat:
    tau_r2 = [x[0]*12 for x in r2_by_mat]
    r2_r2  = [x[1] for x in r2_by_mat]
    axes[1, 0].plot(tau_r2, r2_r2, 'o-', color='steelblue', lw=2, ms=8)
    axes[1, 0].set_xlabel('Maturité (mois)')
    axes[1, 0].set_ylabel('R² moyen')
    axes[1, 0].set_title('R² de la régression par maturité')
    axes[1, 0].axhline(0.5, ls='--', color='firebrick', lw=1.5, label='R²=50%')
    axes[1, 0].legend()

# ─── Résidus : part autonome vs VIX/conditions de marché ────
if decomp_today is not None:
    # Distribution de la part autonome historique
    auto_series = mat_ts.diff().dropna().iloc[:, atm_idx] - \
                  (beta_df.iloc[:, atm_idx] *
                   np.log(spot_ts).diff().dropna().reindex(beta_df.index, method='nearest'))
    auto_series = auto_series.dropna()

    axes[1, 1].hist(auto_series.values * 100, bins=60,
                    color='forestgreen', alpha=0.8, density=True, edgecolor='k', lw=0.3)
    x_auto = np.linspace(auto_series.min()*100, auto_series.max()*100, 200)
    axes[1, 1].plot(x_auto, norm.pdf(x_auto, 0., auto_series.std()*100),
                    'firebrick', lw=2, label='Normale')
    axes[1, 1].set_xlabel('Part autonome Δσ ATM (vol pts × 100)')
    axes[1, 1].set_title(f'Distribution du choc autonome\n'
                          f'(std={auto_series.std()*100:.3f} vol pts)')
    axes[1, 1].legend()

# ─── Synthèse comparaison fenêtres ──────────────────────────
for win_label, color in [('1M','steelblue'),('3M','firebrick'),('6M','forestgreen')]:
    beta_w = BETA_RESULTS[win_label]['beta']
    r2_w   = BETA_RESULTS[win_label]['r2']
    r2_atm_w = r2_w[1.00].dropna()
    axes[1, 2].plot(r2_atm_w.rolling(21).mean().index,
                    r2_atm_w.rolling(21).mean().values,
                    color=color, lw=2, label=f'Fenêtre {win_label}')

axes[1, 2].axhline(0.5, ls='--', color='k', lw=1.5)
axes[1, 2].set_ylabel('R² ATM (moy. 1M)')
axes[1, 2].set_title('R² ATM selon la fenêtre')
axes[1, 2].legend(fontsize=9)
axes[1, 2].xaxis.set_tick_params(rotation=30)

plt.suptitle(f'Section 11 — Validation & Robustesse ({TICKER_FOCUS})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  BILAN QUANTITATIF FINAL
# ============================================================
print('=' * 68)
print('  BILAN — STICKY REGIME DECOMPOSITION')
print('  Actions US | Méthode : Régression glissante + Normalisation SSR')
print('=' * 68)

print(f'''
  MÉTHODOLOGIE
  ──────────────────────────────────────────────────────────────
  1. Lissage non-paramétrique (Nadaraya-Watson) des surfaces brutes
     sur la grille fixe m ∈ {list(M_GRID)} × τ ∈ {[f"{t*12:.0f}M" for t in TAU_GRID]}

  2. Régression glissante (fenêtre 1M/3M/6M) :
     Δσ(m,t) = α(m) + β(m)·ΔlnS(t) + ε(m,t)
     → β(m) = réponse de la vol au spot = signature du régime

  3. Normalisation SSR : β̃(m) = β(m) / |dσ/dlnK|_ATM
     β̃ = 0 → Sticky Delta | β̃ = 1 → Sticky Strike | β̃ = 2 → Sticky Skew

  4. Décomposition polynomiale β(m) = w₀ + w₁(m-1) + w₂(m-1)²
     w₀ : niveau | w₁ : asymétrie puts/calls | w₂ : courbure

  5. Décomposition quotidienne : Δσ = β·ΔlnS (spot) + résidu (autonome)

  POURQUOI ÇA FONCTIONNE MIEUX QUE L'OLS JOUR-À-JOUR
  ──────────────────────────────────────────────────────────────
  • OLS 1 jour : R² s\'effondre quand |ΔS| ≈ 0
  • Fenêtre glissante : β(m) stable car agrège de nombreux jours
  • Sticky Delta (β=0) devient le "niveau de base" naturellement
  • La part autonome est maintenant identifiée et mesurée
''')

print('  RÉSULTATS PAR TICKER (dernière date disponible)')
print('  ──────────────────────────────────────────────────────────────')
print(f'  {"Ticker":<8} {"SSR ATM":<12} {"SSR hist":<12} {"Régime":<20} {"R² moy"}')
print('  ' + '-'*60)
for ticker, res in MULTI_TICKER.items():
    regime = ('Sticky Delta' if res['ssr_atm'] < 0.5 else
              'Sticky Strike' if res['ssr_atm'] < 1.5 else 'Sticky Skew')
    print(f'  {ticker:<8} {res["ssr_atm"]:<12.4f} {res["ssr_atm_mean"]:<12.4f} '
          f'{regime:<20} {res["r2_mean"]:.4f}')

print('\n  INTERPRÉTATION DU SSR')
print('  ──────────────────────────────────────────────────────────────')
print('  β̃ < 0.5  → Vol peu sensible au spot (Sticky Delta)')
print('  β̃ ≈ 1.0  → Vols strikes fixes (Sticky Strike)')
print('  β̃ ≈ 1.5  → Régime intermédiaire typique des marchés equity')
print('  β̃ > 1.5  → Fort effet levier (Sticky Skew)')
print('  β̃ > 2.0  → Au-delà de Sticky Skew (local vol-like)')
print('=' * 68)

---

## Guide d'utilisation pour le trader

### Comment utiliser ce notebook au quotidien

1. **Changer le ticker et la maturité** dans la cellule de la Section 10 :
   ```python
   TICKER_TRADE = 'NVDA'
   TAU_TRADE    = 1/12.   # 1 mois
   WINDOW_TRADE = '3M'
   ```
   et relancer `trader_report(TICKER_TRADE, TAU_TRADE, WINDOW_TRADE)`.

2. **Interpréter le SSR β̃(m=1)** :
   - **β̃ ≈ 0** : Le marché est en régime Sticky Delta — la surface ne bouge pas avec le spot. Le delta BS est le bon.
   - **β̃ ≈ 1** : Régime Sticky Strike — les vols aux strikes fixes sont constantes. Delta à ajuster vers le bas par rapport au Local Vol.
   - **β̃ ≈ 2** : Régime Sticky Skew (Local Vol-like) — la vol ATM se déplace de 2×skew×ΔS. Delta plus agressif.

3. **Regarder la forme de β(m)** :
   - Si β(m) est croissant en m → les calls OTM réagissent plus que les puts OTM
   - Si β(m) est décroissant → effet levier classique (puts OTM réagissent plus)
   - Si β(m) est plat → régime homogène sur tout le smile

4. **Décomposition quotidienne** : la part autonome mesure les chocs de vol non expliqués par le spot — c'est le risque Vega "pur".

---
*Notebook réalisé pour l'analyse du régime de stickiness — Anthropic Claude*